In [ ]:
# SIAKTERNAK Database Module Guide
Dokumentasi lengkap untuk modul `database.py` dalam aplikasi `siakternak`. Panduan ini menjelaskan setiap bagian kode, struktur tabel SQLite, sinkronisasi inventaris, fungsi akuntansi, dan alur pesanan.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 1. Import and Initialize Database Module
Modul ini menggunakan `sqlite3`, `hashlib`, `datetime`, `os`, dan `openpyxl` untuk manajemen data dan ekspor Excel.
</VSCode.Cell>
<VSCode.Cell language="python">
import sqlite3
import hashlib
from datetime import datetime
import os
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

DB_NAME = 'siakternak.db'

def get_connection():
    return sqlite3.connect(DB_NAME)
</VSCode.Cell>
<VSCode.Cell language="markdown">
### init_db()
Fungsi ini membuat tabel SQLite jika belum ada, memeriksa dan menjalankan migrasi schema, menormalisasi data lama, dan menyisipkan data default jika tabel kosong.
</VSCode.Cell>
<VSCode.Cell language="python">
def init_db():
    conn = get_connection()
    c = conn.cursor()

    # Table for users
    c.execute('''CREATE TABLE IF NOT EXISTS users 
                 (id INTEGER PRIMARY KEY AUTOINCREMENT, 
                  username TEXT UNIQUE, password TEXT, 
                  nama_lengkap TEXT, created_at TEXT)''')

    # ... tabel lainnya dibuat di sini ...

    conn.commit()
    conn.close()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 2. Database Schema and Dynamic Migrations
Struktur tabel yang dibuat oleh `init_db()`:
- `users`
- `pemasukan`
- `pengeluaran`
- `inventaris`
- `pesanan`
</VSCode.Cell>
<VSCode.Cell language="python">
# Contoh definisi tabel pemasukan dan pengeluaran
c.execute('''CREATE TABLE IF NOT EXISTS pemasukan 
             (id INTEGER PRIMARY KEY AUTOINCREMENT, 
              jumlah_sapi INTEGER, 
              total_harga INTEGER, 
              tanggal TEXT)''')

c.execute('''CREATE TABLE IF NOT EXISTS pengeluaran 
             (id INTEGER PRIMARY KEY AUTOINCREMENT, 
              produk TEXT, 
              kategori TEXT, 
              nominal INTEGER, 
              jumlah INTEGER DEFAULT 1,
              tanggal TEXT)''')
</VSCode.Cell>
<VSCode.Cell language="markdown">
### Dynamic Migrations
`init_db()` menggunakan `PRAGMA table_info()` untuk mendeteksi kolom yang hilang dan menjalankan `ALTER TABLE` jika diperlukan.
</VSCode.Cell>
<VSCode.Cell language="python">
c.execute("PRAGMA table_info(pengeluaran)")
cols_pengeluaran = [col[1] for col in c.fetchall()]
if 'jumlah' not in cols_pengeluaran:
    c.execute("ALTER TABLE pengeluaran ADD COLUMN jumlah INTEGER DEFAULT 1")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 3. Date Filter Utilities
Fungsi ini digunakan untuk memfilter transaksi berdasarkan bulan dan tahun pada laporan akuntansi.
</VSCode.Cell>
<VSCode.Cell language="python">
def is_in_period(date_str, bulan, tahun):
    if not date_str or len(date_str) < 10:
        return False
    tx_month = date_str[3:5]
    tx_year = date_str[6:10]
    match_month = (bulan == "Semua" or tx_month == bulan)
    match_year = (tahun == "Semua" or tx_year == tahun)
    return match_month and match_year


def is_before_period(date_str, bulan, tahun):
    if not date_str or len(date_str) < 10:
        return False
    tx_month = int(date_str[3:5])
    tx_year = int(date_str[6:10])
    if tahun != "Semua":
        filter_year = int(tahun)
        if tx_year < filter_year:
            return True
        if tx_year > filter_year:
            return False
    if bulan != "Semua":
        filter_month = int(bulan)
        if tx_month < filter_month:
            return True
        if tx_month > filter_month:
            return False
    return False
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 4. User Management Functions
Fungsi-fungsi berikut menangani autentikasi dan manajemen pengguna.
</VSCode.Cell>
<VSCode.Cell language="python">
def verify_user(username, password):
    hashed_password = hashlib.sha256(password.encode()).hexdigest()
    conn = get_connection()
    c = conn.cursor()
    c.execute("SELECT * FROM users WHERE username=? AND password=?", (username, hashed_password))
    user = c.fetchone()
    conn.close()
    return user


def register_user(username, full_name, password):
    hashed_password = hashlib.sha256(password.encode()).hexdigest()
    tgl = datetime.now().strftime("%d/%m/%Y %H:%M")
    conn = get_connection()
    c = conn.cursor()
    try:
        c.execute("INSERT INTO users (username, password, nama_lengkap, created_at) VALUES (?, ?, ?, ?)",
                  (username, hashed_password, full_name, tgl))
        conn.commit()
        success = True
    except sqlite3.IntegrityError:
        success = False
    conn.close()
    return success
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 5. Pemasukan CRUD with Inventaris Sync
Setiap perubahan di `pemasukan` otomatis disinkronkan ke `inventaris`.
</VSCode.Cell>
<VSCode.Cell language="python">
def add_pemasukan(jumlah_sapi, total_harga, tanggal=None):
    if not tanggal:
        tanggal = datetime.now().strftime("%d/%m/%Y %H:%M")
    conn = get_connection()
    c = conn.cursor()
    c.execute("INSERT INTO pemasukan (jumlah_sapi, total_harga, tanggal) VALUES (?, ?, ?)",
              (int(jumlah_sapi), int(total_harga), tanggal))
    pemasukan_id = c.lastrowid
    c.execute("INSERT INTO inventaris (nama_barang, tipe_transaksi, jumlah, tanggal, keterangan, pemasukan_id) VALUES (?, ?, ?, ?, ?, ?)",
              ("Sapi", "Keluar", int(jumlah_sapi), tanggal, f"Penjualan otomatis (Transaksi ID {pemasukan_id})", pemasukan_id))
    conn.commit()
    conn.close()
</VSCode.Cell>
<VSCode.Cell language="markdown">
### Update dan Delete Pemasukan
Fungsi update dan delete juga memperbarui atau menghapus baris terkait di `inventaris`.
</VSCode.Cell>
<VSCode.Cell language="python">
def update_pemasukan(db_id, jumlah_sapi, total_harga):
    conn = get_connection()
    c = conn.cursor()
    c.execute("UPDATE pemasukan SET jumlah_sapi=?, total_harga=? WHERE id=?",
              (int(jumlah_sapi), int(total_harga), int(db_id)))
    c.execute("UPDATE inventaris SET jumlah=? WHERE pemasukan_id=?", (int(jumlah_sapi), int(db_id)))
    conn.commit()
    conn.close()


def delete_pemasukan(db_id):
    conn = get_connection()
    c = conn.cursor()
    c.execute("DELETE FROM pemasukan WHERE id=?", (int(db_id),))
    c.execute("DELETE FROM inventaris WHERE pemasukan_id=?", (int(db_id),))
    conn.commit()
    conn.close()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 6. Pengeluaran CRUD with Inventaris Sync
Mirip dengan pemasukan, pengeluaran otomatis mencatat pembelian ke inventaris.
</VSCode.Cell>
<VSCode.Cell language="python">
def add_pengeluaran(produk, kategori, nominal, jumlah=1, tanggal=None):
    if not tanggal:
        tanggal = datetime.now().strftime("%d/%m/%Y %H:%M")
    conn = get_connection()
    c = conn.cursor()
    c.execute("INSERT INTO pengeluaran (produk, kategori, nominal, jumlah, tanggal) VALUES (?, ?, ?, ?, ?)",
              (produk, kategori, int(nominal), int(jumlah), tanggal))
    pengeluaran_id = c.lastrowid
    item_name = produk
    c.execute("INSERT INTO inventaris (nama_barang, tipe_transaksi, jumlah, tanggal, keterangan, pengeluaran_id) VALUES (?, ?, ?, ?, ?, ?)",
              (item_name, "Masuk", int(jumlah), tanggal, f"Pembelian otomatis (Pengeluaran ID {pengeluaran_id})", pengeluaran_id))
    conn.commit()
    conn.close()
</VSCode.Cell>
<VSCode.Cell language="markdown">
### Update dan Delete Pengeluaran
Perubahan pengeluaran juga memperbarui inventaris berdasarkan `pengeluaran_id`.
</VSCode.Cell>
<VSCode.Cell language="python">
def update_pengeluaran(db_id, produk, kategori, nominal, jumlah=1):
    conn = get_connection()
    c = conn.cursor()
    c.execute("UPDATE pengeluaran SET produk=?, kategori=?, nominal=?, jumlah=? WHERE id=?",
              (produk, kategori, int(nominal), int(jumlah), int(db_id)))
    c.execute("UPDATE inventaris SET nama_barang=?, jumlah=? WHERE pengeluaran_id=?",
              (produk, int(jumlah), int(db_id)))
    conn.commit()
    conn.close()


def delete_pengeluaran(db_id):
    conn = get_connection()
    c = conn.cursor()
    c.execute("DELETE FROM pengeluaran WHERE id=?", (int(db_id),))
    c.execute("DELETE FROM inventaris WHERE pengeluaran_id=?", (int(db_id),))
    conn.commit()
    conn.close()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 7. Manual Inventaris CRUD and Stock Summary
Fungsi inventaris manual memungkinkan input mutasi inventaris yang tidak terkait langsung dengan pemasukan atau pengeluaran.
</VSCode.Cell>
<VSCode.Cell language="python">
def add_inventaris_manual(nama_barang, tipe_transaksi, jumlah, keterangan, tanggal=None):
    if not tanggal:
        tanggal = datetime.now().strftime("%d/%m/%Y %H:%M")
    conn = get_connection()
    c = conn.cursor()
    c.execute("INSERT INTO inventaris (nama_barang, tipe_transaksi, jumlah, tanggal, keterangan) VALUES (?, ?, ?, ?, ?)",
              (nama_barang, tipe_transaksi, int(jumlah), tanggal, keterangan))
    conn.commit()
    conn.close()


def get_all_inventaris():
    conn = get_connection()
    c = conn.cursor()
    c.execute("SELECT id, nama_barang, tipe_transaksi, jumlah, tanggal, keterangan FROM inventaris ORDER BY id DESC")
    rows = c.fetchall()
    conn.close()
    return rows
</VSCode.Cell>
<VSCode.Cell language="markdown">
### get_inventory_summary()
Fungsi ini menghitung stok untuk `Sapi`, `Pakan`, dan `Obat` dengan menjumlahkan `Masuk` dan `Keluar`.
</VSCode.Cell>
<VSCode.Cell language="python">
def get_inventory_summary():
    conn = get_connection()
    c = conn.cursor()
    c.execute("SELECT SUM(jumlah) FROM inventaris WHERE nama_barang='Sapi' AND tipe_transaksi='Masuk'")
    sapi_in = c.fetchone()[0] or 0
    c.execute("SELECT SUM(jumlah) FROM inventaris WHERE nama_barang='Sapi' AND tipe_transaksi='Keluar'")
    sapi_out = c.fetchone()[0] or 0
    sapi_stock = sapi_in - sapi_out
    
    c.execute("SELECT SUM(jumlah) FROM inventaris WHERE nama_barang LIKE '%Pakan%' AND tipe_transaksi='Masuk'")
    pakan_in = c.fetchone()[0] or 0
    c.execute("SELECT SUM(jumlah) FROM inventaris WHERE nama_barang LIKE '%Pakan%' AND tipe_transaksi='Keluar'")
    pakan_out = c.fetchone()[0] or 0
    pakan_stock = pakan_in - pakan_out
    
    c.execute("SELECT SUM(jumlah) FROM inventaris WHERE (nama_barang LIKE '%Obat%' OR nama_barang LIKE '%Vaksin%' OR nama_barang LIKE '%Kesehatan%') AND tipe_transaksi='Masuk'")
    obat_in = c.fetchone()[0] or 0
    c.execute("SELECT SUM(jumlah) FROM inventaris WHERE (nama_barang LIKE '%Obat%' OR nama_barang LIKE '%Vaksin%' OR nama_barang LIKE '%Kesehatan%') AND tipe_transaksi='Keluar'")
    obat_out = c.fetchone()[0] or 0
    obat_stock = obat_in - obat_out
    
    conn.close()
    return {
        'sapi': sapi_stock,
        'pakan': pakan_stock,
        'obat': obat_stock
    }
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 8. Accounting Helper Functions and Financial Reports
Bagian ini menghasilkan laporan akuntansi berbasis data pemasukan dan pengeluaran.
</VSCode.Cell>
<VSCode.Cell language="python">
def get_coa():
    return [
        ("101", "Kas & Bank", "Aset"),
        ("102", "Persediaan Sapi Bakalan", "Aset"),
        ("103", "Persediaan Pakan", "Aset"),
        ("104", "Persediaan Obat", "Aset"),
        ("401", "Pendapatan Penjualan Sapi", "Pendapatan"),
        ("501", "Harga Pokok Penjualan Sapi", "HPP"),
        ("502", "Beban Transportasi Pembelian", "HPP"),
        ("601", "Beban Pakan", "Beban"),
        ("602", "Beban Kesehatan Ternak", "Beban"),
        ("603", "Beban Gaji", "Beban"),
        ("604", "Beban Listrik & Air Kandang", "Beban"),
        ("605", "Beban Penyusutan Kandang & Alat", "Beban"),
        ("606", "Beban Operasional Lainnya", "Beban")
    ]
</VSCode.Cell>
<VSCode.Cell language="markdown">
### map_expense_account()
Fungsi ini menentukan akun yang benar berdasarkan `produk` dan `kategori` pengeluaran.
</VSCode.Cell>
<VSCode.Cell language="python">
def map_expense_account(produk, kategori):
    cat_lower = kategori.lower()
    prod_lower = produk.lower()
    if "pakan" in cat_lower or "pakan" in prod_lower:
        return "103", "Persediaan Pakan"
    elif "kesehatan" in cat_lower or "obat" in cat_lower or "vaksin" in prod_lower:
        return "104", "Persediaan Obat"
    elif "gaji" in cat_lower or "gaji" in prod_lower:
        return "603", "Beban Gaji"
    elif "listrik" in cat_lower or "air" in cat_lower or "listrik" in prod_lower or "air" in prod_lower:
        return "604", "Beban Listrik & Air Kandang"
    elif "penyusutan" in cat_lower or "penyusutan" in prod_lower:
        return "605", "Beban Penyusutan Kandang & Alat"
    elif "transport" in cat_lower or "angkut" in cat_lower or "transport" in prod_lower or "angkut" in prod_lower:
        return "502", "Beban Transportasi Pembelian"
    elif "sapi" in cat_lower or "bakalan" in cat_lower or "sapi" in prod_lower or "bakalan" in prod_lower:
        return "102", "Persediaan Sapi Bakalan"
    else:
        return "606", "Beban Operasional Lainnya"
</VSCode.Cell>
<VSCode.Cell language="markdown">
### get_jurnal_umum()
Fungsi ini membuat entri jurnal umum berikut:
- Debit Kas
- Kredit Pendapatan
- Debit HPP
- Kredit Persediaan Sapi Bakalan
- dan penyesuaian untuk pakan/obat
</VSCode.Cell>
<VSCode.Cell language="python">
def get_jurnal_umum(bulan="Semua", tahun="Semua"):
    conn = get_connection()
    c = conn.cursor()
    c.execute("SELECT jumlah_sapi, total_harga, tanggal FROM pemasukan")
    pemasukan_rows = c.fetchall()
    c.execute("SELECT produk, kategori, nominal, tanggal FROM pengeluaran")
    pengeluaran_rows = c.fetchall()
    conn.close()
    # ... gabungkan transaksi lalu susun jurnal ...
</VSCode.Cell>
<VSCode.Cell language="markdown">
### get_buku_besar(), get_neraca_saldo(), get_laba_rugi()
Fungsi-fungsi ini mengonversi transaksi menjadi laporan buku besar, neraca saldo, dan laba rugi.
</VSCode.Cell>
<VSCode.Cell language="python">
def get_laba_rugi(bulan="Semua", tahun="Semua"):
    conn = get_connection()
    c = conn.cursor()
    c.execute("SELECT total_harga, tanggal FROM pemasukan")
    pemasukan_rows = c.fetchall()
    c.execute("SELECT produk, kategori, nominal, tanggal FROM pengeluaran")
    pengeluaran_rows = c.fetchall()
    conn.close()
    pb = {"401": 0, "501": 0, "502": 0, "601": 0, "602": 0, "603": 0, "604": 0, "605": 0, "606": 0}
    for val, tgl in pemasukan_rows:
        if is_in_period(tgl, bulan, tahun):
            pb["401"] += val
            pb["501"] += int(val * 0.7)
    for prod, cat, val, tgl in pengeluaran_rows:
        if is_in_period(tgl, bulan, tahun):
            code, _ = map_expense_account(prod, cat)
            if code == "103":
                pb["601"] += int(val * 0.8)
            elif code == "104":
                pb["602"] += int(val * 0.8)
            elif code in pb:
                pb[code] += val
    total_revenue = pb["401"]
    total_hpp = pb["501"] + pb["502"]
    total_beban = pb["601"] + pb["602"] + pb["603"] + pb["604"] + pb["605"] + pb["606"]
    laba_bersih = total_revenue - total_hpp - total_beban
    return {
        'pendapatan': total_revenue,
        'hpp_sapi': pb['501'],
        'beban_transport': pb['502'],
        'total_hpp': total_hpp,
        'beban_pakan': pb['601'],
        'beban_kesehatan': pb['602'],
        'beban_gaji': pb['603'],
        'beban_listrik': pb['604'],
        'beban_penyusutan': pb['605'],
        'beban_lain': pb['606'],
        'total_beban': total_beban,
        'laba_bersih': laba_bersih
    }
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 9. Buyer Orders (Pesanan) and Inventory Sync
Bagian ini mengelola pesanan yang dibuat di layar landing dan mempengaruhi inventaris jika pesanan disetujui.
</VSCode.Cell>
<VSCode.Cell language="python">
def add_pesanan(nama, wa, jenis_sapi, jumlah_sapi, keterangan, tanggal=None):
    if not tanggal:
        tanggal = datetime.now().strftime("%d/%m/%Y %H:%M")
    conn = get_connection()
    c = conn.cursor()
    c.execute("INSERT INTO pesanan (nama, wa, jenis_sapi, jumlah_sapi, keterangan, status, tanggal) VALUES (?, ?, ?, ?, ?, ?, ?)",
              (nama, wa, jenis_sapi, int(jumlah_sapi), keterangan, 'Pending', tanggal))
    conn.commit()
    conn.close()


def update_status_pesanan(pesanan_id, status):
    conn = get_connection()
    c = conn.cursor()
    c.execute("SELECT nama, jumlah_sapi, tanggal FROM pesanan WHERE id = ?", (int(pesanan_id),))
    row = c.fetchone()
    if not row:
        conn.close()
        return False
    nama, jumlah_sapi, tanggal = row
    c.execute("UPDATE pesanan SET status = ? WHERE id = ?", (status, int(pesanan_id)))
    if status == 'Iya':
        c.execute("SELECT id FROM inventaris WHERE pesanan_id = ?", (int(pesanan_id),))
        inv_row = c.fetchone()
        if not inv_row:
            c.execute("INSERT INTO inventaris (nama_barang, tipe_transaksi, jumlah, tanggal, keterangan, pesanan_id) VALUES (?, ?, ?, ?, ?, ?)",
                      ("Sapi", "Keluar", int(jumlah_sapi), tanggal, f"Acc Pemesanan ID {pesanan_id} ({nama})", int(pesanan_id)))
    else:
        c.execute("DELETE FROM inventaris WHERE pesanan_id = ?", (int(pesanan_id),))
    conn.commit()
    conn.close()
    return True
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 10. Exporting Data for Word-Compatible Output
Aplikasi menyediakan `export_to_excel()` untuk mengekspor data ke Excel. Data Excel ini dapat digunakan sebagai dasar untuk laporan Word.
</VSCode.Cell>
<VSCode.Cell language="python">
def export_to_excel(filepath="Laporan_Siakternak.xlsx"):
    wb = openpyxl.Workbook()
    ws_in = wb.active
    ws_in.title = "Pemasukan"
    ws_in.append(["No", "Jumlah Sapi (Ekor)", "Total Penjualan (Rp)", "Tanggal"])
    rows_in = get_all_pemasukan()
    for i, r in enumerate(reversed(rows_in), start=1):
        ws_in.append([i, r[1], r[2], r[3]])
    style_sheet(ws_in, "A1:D1", ["#FFD6D6D6", "#FF00FF00"])
    wb.save(filepath)
    return os.path.abspath(filepath)
</VSCode.Cell>
<VSCode.Cell language="python">
def style_sheet(ws, header_range, colors):
    header_font = Font(name="Arial", size=11, bold=True, color="FFFFFF")
    if "00FF00" in colors[1]:
        fill_color = "1B5E20"
    elif "FF0000" in colors[1]:
        fill_color = "B71C1C"
    else:
        fill_color = "1A237E"
    header_fill = PatternFill(start_color=fill_color, end_color=fill_color, fill_type="solid")
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center")
    thin_border = Border(
        left=Side(style='thin', color='DDDDDD'),
        right=Side(style='thin', color='DDDDDD'),
        top=Side(style='thin', color='DDDDDD'),
        bottom=Side(style='thin', color='DDDDDD')
    )
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.border = thin_border
            if isinstance(cell.value, int):
                cell.number_format = '#,##0'
                cell.alignment = Alignment(horizontal="right")
            elif cell.column == 1:
                cell.alignment = Alignment(horizontal="center")
            else:
                cell.alignment = Alignment(horizontal="left")
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = openpyxl.utils.get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 12)
</VSCode.Cell>
